# Metodologia descriptiva de clientes y estructuras LGF

Este notebook documenta el modulo descriptivo vigente del proyecto. Su objetivo es
explicar como se transforma la base historica acumulada en tablas consultables para
el **Visualizador general de clientes**, la vista **Estructuras y componentes** y la
entrada controlada del modulo de clusters.

El notebook es de estudio y auditoria: lee salidas ya generadas. La ejecucion
oficial se realiza con `run_descriptivos.py`.

## 1. Arquitectura activa y fuente oficial

```text
bases de datos historicas/historic_sales_acum.csv
  (con campos originales de receta y estructura)
             |
             | clasificar tipologia directamente desde fuente
             v
run_descriptivos.py
             |
             v
resultados/descriptivos/
  historico_confirmado.csv              -> estructuras y entrada clusters (lineas con tallos)
  historico_visualizador_comercial.csv  -> visualizador (incluye valor sin tallos)
  perfil_cliente.csv                     -> caracterizacion por cliente
  ventas_semana_cliente_producto.csv     -> evolucion comercial
  cliente_sku_operativo_resumen.csv      -> orden recurrente
  cliente_sku_operativo_composicion.csv  -> componentes del SKU
  estructura_caja.csv                    -> estructura regular compacta
  estructura_componentes.csv             -> componentes compactos
  catalogo_estructura_version.csv        -> versiones observadas
```

La fuente oficial es el archivo acumulado. Los archivos anuales pueden conservarse
como respaldo, pero no deben concatenarse adicionalmente si ya estan contenidos en
`historic_sales_acum.csv`, porque se duplicarian pedidos.

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

RAW_PATH = ROOT / "bases de datos historicas" / "historic_sales_acum.csv"
OUTPUT_DIR = ROOT / "resultados" / "descriptivos"

def read_output(name, usecols=None, parse_dates=None):
    path = OUTPUT_DIR / f"{name}.csv"
    if not path.exists():
        raise FileNotFoundError(f"No existe {path}. Ejecuta primero run_descriptivos.py.")
    return pd.read_csv(path, usecols=usecols, parse_dates=parse_dates, low_memory=False)

print("Fuente:", RAW_PATH)
print("Existe fuente:", RAW_PATH.exists())
print("Output descriptivo:", OUTPUT_DIR)

## 2. Limpieza y regla de tipo de pedido

La funcion central es `src/lgf_operativo/cleaning.py::clean_historical_orders()`.
Normaliza columnas, fechas, cliente, pais, estado, cantidades, ventas, productos,
colores y llaves operativas.

La clasificacion `tipo_pedido_operativo` sigue esta lectura:

| Tipo visible | Lectura analitica | Uso comercial |
|---|---|---|
| `SOLIDO` | Producto-color terminado | Puede alimentar el forecast de solidos |
| `SURTIDO` | Mezcla regular de componentes | Se analiza como composicion |
| `SURTIDO_M` | Mezcla especial de componentes | Se analiza como composicion |
| `RAINBOW` | Receta de colores | Se analiza como composicion |
| `BOUQUET` | Bouquet/receta historica | Se analiza como estructura mixta |
| `BQT` | Marca explicita BQT | Se analiza como estructura mixta |
| `COMBO` | Combo de componentes | Se analiza como estructura mixta |
| `BULK` | Volumen a granel | Lectura operativa separada |

**Regla critica:** las marcas explicitas `BQT`, `COMBO` y `RAINBOW`
prevalecen sobre una marca generica `Solido`. `BOUQUET` permanece como
categoria historica separada, igualmente mixta; no se convierte masivamente
en `BQT`.

La base acumulada oficial debe conservar los campos originales de receta y
estructura. Con esos campos, la clasificacion se hace directamente desde la
fuente; `BQT`, `COMBO` y `RAINBOW` prevalecen sobre marcas genericas de
`Solido`. Una referencia historica existe solo como respaldo de contingencia
para extractos reducidos y no pertenece al flujo normal.

In [ ]:
historico_cols = [
    "fecha", "cod_cliente", "cliente", "pais", "tipo_pedido_operativo",
    "familia_analisis_operativa", "origen_tipologia_operativa", "producto", "color", "tallos_confirmados"
]
historico = read_output("historico_confirmado", usecols=historico_cols, parse_dates=["fecha"])

resumen_tipo = (
    historico.groupby("tipo_pedido_operativo", dropna=False, as_index=False)
    .agg(
        lineas=("cod_cliente", "size"),
        clientes=("cod_cliente", "nunique"),
        tallos=("tallos_confirmados", "sum"),
    )
    .sort_values("tallos", ascending=False)
)
resumen_tipo

# Auditoria del origen de tipologia. Con la base completa, la ejecucion normal
# debe concentrarse en reglas de campos fuente y marcas explicitas.
historico.groupby("origen_tipologia_operativa", dropna=False, as_index=False).agg(
    lineas=("cod_cliente", "size"),
    tallos=("tallos_confirmados", "sum"),
)

## 3. Perfiles, SKUs y tablas de visualizacion

El descriptivo separa pedidos confirmados de pendientes o cambios. Las
caracterizaciones y los clusters usan el historico confirmado con movimiento fisico
de tallos para evitar mezclar intenciones o registros puramente contables.
El visualizador usa adicionalmente `historico_visualizador_comercial.csv`, que
conserva lineas monetarias sin tallos cuando el valor fue registrado separado de
los componentes de la receta; asi calcula Ventas USD y USD/tallo correctamente.

| Salida | Pregunta que responde |
|---|---|
| `perfil_cliente.csv` | Cuanto compra, con que frecuencia y que tan concentrado es cada cliente |
| `mix_*` | Cual es la composicion por producto, color, formato o SKU |
| `cliente_sku_operativo_resumen.csv` | Que estructura repite normalmente el cliente |
| `cliente_sku_operativo_composicion.csv` | Que componentes integran cada estructura |
| `ventas_*_periodo.csv` | Como se comportan volumen y ventas en el tiempo |

Para `SOLIDO`, el color define el SKU. Para `SURTIDO`, `SURTIDO_M`, `RAINBOW`,
`BOUQUET`, `BQT` y `COMBO`, el color es un componente interno de una estructura o receta.

In [ ]:
perfil = read_output("perfil_cliente")
sku_resumen = read_output("cliente_sku_operativo_resumen")

perfil.head(10), sku_resumen.head(10)

## 4. Estructuras y componentes: salida compacta vigente

La version inicial materializaba una fila por linea historica y generaba archivos
de gran tamano. La implementacion actual de
`src/lgf_operativo/metrics.py::build_operational_structure_tables()` resume por:

```text
cliente + semana + tipo de pedido + SKU operativo + version de composicion
```

Esto conserva:

- Tallos totales de la estructura.
- Cantidad real de repeticiones observadas (`repeticiones_estructura`).
- Componentes por producto, color y variedad.
- Numero de estructuras que contienen cada componente (`estructuras_componente`).
- Versiones de composicion para identificar una orden regular.

No conserva cada linea repetida como fila del dashboard, porque el detalle
transaccional completo permanece en `historico_confirmado.csv`.

Esta compactacion mantiene los filtros por cliente, ano, semana, producto y color
de la pestana **Estructuras y componentes**.

In [ ]:
estructura_cols = [
    "cod_cliente", "cliente", "anio_semana", "tipo_pedido_operativo",
    "sku_operativo", "composicion_version_id", "repeticiones_estructura",
    "tallos_estructura", "productos", "colores"
]
componente_cols = [
    "cod_cliente", "cliente", "anio_semana", "tipo_pedido_operativo",
    "sku_operativo", "producto", "color", "variedad",
    "estructuras_componente", "tallos_analisis"
]

estructura_caja = read_output("estructura_caja", usecols=estructura_cols)
estructura_componentes = read_output("estructura_componentes", usecols=componente_cols)

estructura_caja.head(10), estructura_componentes.head(10)

In [ ]:
# Auditoria opcional de conservacion de tallos.
# Puede consumir memoria sobre toda la historia; activala solo cuando se requiera validar una corrida.
EJECUTAR_AUDITORIA_TOTALES = False

if EJECUTAR_AUDITORIA_TOTALES:
    tallos_hist = pd.read_csv(
        OUTPUT_DIR / "historico_confirmado.csv",
        usecols=["tallos_analisis"],
        low_memory=False,
    )["tallos_analisis"].sum()
    tallos_estructura = estructura_caja["tallos_estructura"].sum()
    tallos_componentes = estructura_componentes["tallos_analisis"].sum()
    auditoria = pd.DataFrame({
        "fuente": ["historico_confirmado", "estructura_caja", "estructura_componentes"],
        "tallos": [tallos_hist, tallos_estructura, tallos_componentes],
    })
    display(auditoria)
    assert tallos_hist == tallos_estructura == tallos_componentes

## 5. Relacion con clusters y forecast

- **Clusters** consume `historico_confirmado.csv`, filtra un ano seleccionado y
  vuelve a calcular atributos del cliente dentro de ese ano.
- **Forecast** no consume las tablas descriptivas para entrenar. Limpia la fuente
  acumulada de forma independiente y conserva solamente `SOLIDO` confirmado.
- Un cambio en la regla de tipos de pedido requiere regenerar descriptivos,
  clusters de los anos consultados y forecast.

## 6. Ejecucion oficial en Bash

```bash
python run_descriptivos.py \
  --historico "bases de datos historicas/historic_sales_acum.csv" \
  --output "resultados/descriptivos" \
  --no-cache
```

Omitir `--year`/`--years` deja disponible toda la historia para el visualizador.

## 7. Lectura para negocio

El descriptivo no toma decisiones por si solo. Prepara evidencia para responder:

1. Que clientes compran regularmente y en que volumen.
2. Cual es su orden base repetida.
3. Si el pedido se compra como producto-color terminado o se arma como receta.
4. Que componentes, colores y productos requieren conversacion comercial u
   organizacion operativa.
5. Que base limpia debe utilizar el modulo de segmentacion.